# Comparing `openicu` and `ricu/yaib` in YAIB wide format

**What was done:**

* We compare the first `7 * 24` hours.
* `openicu` is already in the target YAIB wide format.
* `openicu_window` was calculated from `openicu` using `min(time)` as `start` and `max(time)` as `end` per `stay_id`.
* `ricu_window` was imported from R/ricu, converted to the same dtypes, and `end` values larger than `7 * 24` were capped to `7 * 24`.
* `ricu` was harmonized to the same columns as `openicu`, joined with `ricu_window`, and filtered by `time >= start` and `time <= end`.

**Current status:**

* No `stay_id`s are missing between `openicu_window` and `ricu_window`.
* For the shared stays, almost all `start`/`end` windows match exactly.
* There are 14 remaining cases where `start` matches, but RICU has `end = 168` while OpenICU only has `end = 0`.

**Observation for one example (`stay_id = 30924165`):**

* RICU contains rows for `time = 0..168`, but all concept values are `null`.
* OpenICU contains only `time = 0`, but with non-null concept values.
* The raw RICU dynamic table contains values for this stay, but they appear outside or differently aligned to the comparison grid.

**Conclusion:**
The main cohort/window alignment is mostly resolved. The remaining issue seems to be an edge-case time/value alignment difference: OpenICU assigns values to `time = 0`, while RICU has no non-null values in the corresponding `0..168` grid for these 14 stays.


# Import and Paths

In [ ]:
import polars as pl
from pathlib import Path

output_path = Path("~/output/openicu_yaib")

# Loading parquets and preprocessing/ harmonication

In [ ]:
MAX_HOURS = 24 * 7

openicu = (
    pl.read_parquet(output_path / "openicu_dyn_all.parquet")
    .filter(pl.col("time") <= MAX_HOURS)
)

openicu.describe()

In [ ]:
openicu_window = openicu.group_by("stay_id").agg([
    pl.col("time").min().alias("start"),
    pl.col("time").max().alias("end"),
]).sort("stay_id")

openicu_window.describe()

In [ ]:
MAX_HOURS = 24 * 7

ricu_window = (
    pl.read_parquet(output_path / "ricu_stay_windows_miiv.parquet")
    .with_columns([
        pl.col("stay_id").cast(pl.Int64),
        (pl.col("start") / 3_600_000).cast(pl.Int64),
        (pl.col("end") / 3_600_000).cast(pl.Int64),
    ])
    .sort("stay_id")
)

ricu_window.write_parquet(output_path / "ricu_window.parquet")

ricu_window = (
    ricu_window
    .with_columns([
        pl.min_horizontal(
            pl.col("end"),
            pl.lit(MAX_HOURS),
        ).alias("end")
    ])
)

ricu_window.describe()

In [ ]:
ricu = pl.read_parquet(output_path / "ricu_dynamic_vars_miiv.parquet")

In [ ]:
ricu = ricu.with_columns([
    pl.col("stay_id").cast(pl.Int64),
    pl.col("time").cast(pl.Int64),
])

# other_cols = [c for c in ricu.columns if c not in {"stay_id", "time", "charttime"}]
# ricu = ricu.select(["stay_id", "time", *other_cols])
# they have the same concepts
ricu = ricu.select(openicu.columns)

ricu.write_parquet(output_path / "ricu.parquet")

ricu = (
    ricu
    .join(ricu_window, on="stay_id", how="inner")
    .filter((pl.col("time") >= pl.col("start")) & (pl.col("time") <= pl.col("end")))
    .select([c for c in ricu.columns])
)

ricu.describe()

# Analysis

## Comparing df `ricu_window` and df `openicu_window`

### Check what `stay_ids` are missing

In [ ]:
# check if cells in column stay_id are only in the df `ricu_window` and not in the df `openicu_window`
openicu_stay_ids = set(openicu_window["stay_id"].unique())
ricu_stay_ids = set(ricu_window["stay_id"].unique())
only_in_ricu = ricu_stay_ids - openicu_stay_ids
print(f"Number of stay_ids only in ricu_window: {len(only_in_ricu)}")
# and vise versa
only_in_openicu = openicu_stay_ids - ricu_stay_ids
print(f"Number of stay_ids only in openicu_window: {len(only_in_openicu)}")

**Conclusion**: No stay_ids are missing

### Check if the `start`and  `end`-columns per `stay_ids` are the same in both df_window.

In [ ]:
# join the two dataframes on stay_id and compare the columns start and end
joined = openicu_window.join(ricu_window, on="stay_id", how="inner", suffix="_ricu")
joined = joined.with_columns([
    (pl.col("start") - pl.col("start_ricu")).alias("start_sign_diff"),
    (pl.col("end") - pl.col("end_ricu")).alias("end_sign_diff"),
])

joined = joined.select(["stay_id", "start_sign_diff", "end_sign_diff"]).filter((pl.col("start_sign_diff") != 0) | (pl.col("end_sign_diff") != 0))

joined.describe()

In [ ]:
ricu_more = joined.filter((pl.col("start_sign_diff") > 0) | (pl.col("end_sign_diff") < 0))
ricu_more

In [ ]:
ricu_more.write_parquet(output_path / "ricu_more.parquet")

In [ ]:
openicu_more = joined.filter((pl.col("start_sign_diff") < 0) | (pl.col("end_sign_diff") > 0))
openicu_more

# Sandbox

In [ ]:
ricu.filter(pl.col("stay_id") == 30924165).describe()

In [ ]:
pl.read_parquet(output_path / "ricu_dynamic_vars_miiv.parquet").filter(pl.col("stay_id") == 30924165).describe()

In [ ]:
openicu.filter(pl.col("stay_id") == 30924165).describe()